# Network Dynamics and Synaptic Plasticity in Spiking Neural Networks

**Authors:** [SASAN MALEKNIA 533394, SINA MALEKNIA 533557]  
**Course:** Brain Modelling - Artificial Intelligence  
**Date:** February 2026

---

## Project Overview

This project extends the basic Leaky Integrate-and-Fire (LIF) neuron model to explore:
1. **Network construction** using NEST simulator
2. **Different connectivity patterns** (sparse vs dense)
3. **Synaptic plasticity** via STDP (Spike-Timing-Dependent Plasticity)
4. **Network dynamics** including synchronization and firing rate evolution

### Mathematical Framework

**LIF Neuron Dynamics:**
$$\tau_m \frac{dV}{dt} = -(V - E_L) + \frac{I_{syn}}{g_L}$$

**STDP Learning Rule:**
$$\Delta w = \begin{cases}
A_+ \cdot e^{-\frac{\Delta t}{\tau_+}} & \text{if } \Delta t > 0 \text{ (pre before post)} \\
A_- \cdot e^{\frac{\Delta t}{\tau_-}} & \text{if } \Delta t < 0 \text{ (post before pre)}
\end{cases}$$

---
## Part 1: Review - Single LIF Neuron (from Handson)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nest

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print(f"NEST version: {nest.__version__}")
nest.ResetKernel()

In [ ]:
# Original handson functions for single neuron
def default_pars(**kwargs):
    pars = {}
    pars['V_th'] = -55.0
    pars['V_reset'] = -75.0
    pars['tau_m'] = 10.0
    pars['g_L'] = 10.0
    pars['V_init'] = -75.0
    pars['E_L'] = -75.0
    pars['tref'] = 2.0
    pars['sim_len'] = 1000.0
    pars['dt'] = 0.1
    pars['t_steps'] = np.arange(0, pars['sim_len'], pars['dt'])
    for k in kwargs:
        pars[k] = kwargs[k]
    return pars

def run_LIF(pars, I, stop=False):
    V_th, V_reset = pars['V_th'], pars['V_reset']
    tau_m, g_L = pars['tau_m'], pars['g_L']
    V_init, E_L = pars['V_init'], pars['E_L']
    dt, t_steps = pars['dt'], pars['t_steps']
    tref = pars['tref']
    t = t_steps.size
    
    v = np.zeros(t)
    v[0] = V_init
    I = I * np.ones(t)
    if stop:
        I[:int(len(I) / 2) - 1000] = 0
        I[int(len(I) / 2) + 1000:] = 0
    tr = 0.0
    
    rec_spikes = []
    for i in range(t - 1):
        if tr > 0:
            v[i] = V_reset
            tr = tr - 1
        elif v[i] >= V_th:
            rec_spikes.append(i)
            v[i] = V_reset
            tr = tref / dt
        
        dv = (-(v[i] - E_L) + I[i] / g_L) * (dt / tau_m)
        v[i + 1] = v[i] + dv
    
    rec_spikes = np.array(rec_spikes) * dt
    return v, rec_spikes

In [ ]:
# Test single LIF neuron
pars = default_pars()
I_input = 250
v, spikes = run_LIF(pars, I_input)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pars['t_steps'], v, 'b-', linewidth=1.5)
ax.axhline(pars['V_th'], color='r', linestyle='--', label='Threshold', linewidth=2)
ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_ylabel('Membrane Potential (mV)', fontsize=12)
ax.set_title(f'Single LIF Neuron Response (I = {I_input} pA)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Number of spikes: {len(spikes)}")
print(f"Firing rate: {len(spikes) / (pars['sim_len'] / 1000):.2f} Hz")

---
## Part 2: Building a Network with NEST

In [ ]:
# Reset and configure NEST
nest.ResetKernel()
nest.resolution = 0.1

# Network parameters
N_neurons = 100
sim_time = 1000.0
p_conn = 0.1

# IMPORTANT: Increased input parameters for reliable spiking
POISSON_RATE = 1500.0  # Hz (increased from 800)
INPUT_WEIGHT = 80.0    # pA (increased from 50)
RECURRENT_WEIGHT = 20.0  # pA

# Neuron parameters
neuron_params = {
    'V_th': -55.0,
    'V_reset': -75.0,
    'tau_m': 10.0,
    'E_L': -75.0,
    't_ref': 2.0,
    'C_m': 250.0,
    'V_m': -75.0
}

print("Parameters configured")
print(f"  - Poisson input rate: {POISSON_RATE} Hz")
print(f"  - Input weight: {INPUT_WEIGHT} pA")

In [ ]:
# Create neurons
neurons = nest.Create('iaf_psc_alpha', N_neurons, params=neuron_params)
print(f"Created {N_neurons} LIF neurons")

In [ ]:
# Create Poisson input generators
poisson_generators = nest.Create('poisson_generator', N_neurons, 
                                  params={'rate': POISSON_RATE})
nest.Connect(poisson_generators, neurons, 'one_to_one',
             syn_spec={'weight': INPUT_WEIGHT, 'delay': 1.0})

print(f"Created external input")
print(f"  - {N_neurons} Poisson generators at {POISSON_RATE} Hz")
print(f"  - Input weight: {INPUT_WEIGHT} pA")

In [ ]:
# Create recurrent connections
nest.Connect(neurons, neurons,
             conn_spec={'rule': 'pairwise_bernoulli', 'p': p_conn},
             syn_spec={'weight': RECURRENT_WEIGHT, 'delay': 1.5})

connections = nest.GetConnections(neurons, neurons)
print(f"Created {len(connections)} recurrent connections (p={p_conn})")

In [ ]:
# Create recording devices
spike_recorder = nest.Create('spike_recorder')
nest.Connect(neurons, spike_recorder)

sample_neurons = neurons[:5]
multimeter = nest.Create('multimeter', params={'record_from': ['V_m'], 'interval': 0.1})
nest.Connect(multimeter, sample_neurons)

print(f"Recording devices created")

In [ ]:
# Run simulation
print(f"Simulating {sim_time} ms...")
nest.Simulate(sim_time)
print("Simulation complete!")

In [ ]:
# Extract and analyze spike data
spike_events = spike_recorder.get('events')
spike_times = spike_events['times']
spike_senders = spike_events['senders']

total_spikes = len(spike_times)
mean_firing_rate = total_spikes / (N_neurons * sim_time / 1000.0)

print(f"\nNetwork Activity Statistics:")
print(f"  - Total spikes: {total_spikes}")
print(f"  - Mean firing rate: {mean_firing_rate:.2f} Hz")
print(f"  - Active neurons: {len(np.unique(spike_senders))} / {N_neurons}")

In [ ]:
# Plot raster and population firing rate
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Raster plot
axes[0].scatter(spike_times, spike_senders, s=1, c='black', alpha=0.5)
axes[0].set_ylabel('Neuron ID', fontsize=12)
axes[0].set_xlabel('Time (ms)', fontsize=12)
axes[0].set_title('Network Raster Plot (Sparse Connectivity, p=0.1)', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, sim_time)
axes[0].grid(True, alpha=0.3)

# Population firing rate
bin_size = 10.0
bins = np.arange(0, sim_time + bin_size, bin_size)
hist, _ = np.histogram(spike_times, bins=bins)
firing_rate_pop = hist / (N_neurons * bin_size / 1000.0)
bin_centers = (bins[:-1] + bins[1:]) / 2

axes[1].plot(bin_centers, firing_rate_pop, 'b-', linewidth=2)
axes[1].axhline(mean_firing_rate, color='r', linestyle='--',
                label=f'Mean = {mean_firing_rate:.2f} Hz', linewidth=2)
axes[1].set_ylabel('Firing Rate (Hz)', fontsize=12)
axes[1].set_xlabel('Time (ms)', fontsize=12)
axes[1].set_title('Population Firing Rate', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Figure 1 complete!")

In [ ]:
# Plot membrane potentials
voltage_events = multimeter.get('events')
vm = voltage_events['V_m']
times = voltage_events['times']
senders = voltage_events['senders']

fig, axes = plt.subplots(len(sample_neurons), 1, figsize=(14, 8), sharex=True)

for i, neuron_id in enumerate(sample_neurons.get('global_id')):
    mask = senders == neuron_id
    axes[i].plot(times[mask], vm[mask], 'b-', linewidth=0.8)
    axes[i].axhline(-55, color='r', linestyle='--', alpha=0.5, linewidth=1)
    axes[i].set_ylabel(f'V_m (mV)\nNeuron {neuron_id}', fontsize=10)
    axes[i].grid(True, alpha=0.3)
    axes[i].set_ylim(-80, -50)

axes[-1].set_xlabel('Time (ms)', fontsize=12)
axes[0].set_title('Membrane Potentials of Sample Neurons', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 3: Implementing STDP

In [ ]:
# Reset for STDP network
nest.ResetKernel()
nest.resolution = 0.1

# Create neurons
neurons_stdp = nest.Create('iaf_psc_alpha', N_neurons, params=neuron_params)

# Create Poisson input (same strong parameters)
poisson_generators_stdp = nest.Create('poisson_generator', N_neurons,
                                       params={'rate': POISSON_RATE})
nest.Connect(poisson_generators_stdp, neurons_stdp, 'one_to_one',
             syn_spec={'weight': INPUT_WEIGHT, 'delay': 1.0})

print("STDP network created")

In [ ]:
# Define STDP synapse
stdp_params = {
    'synapse_model': 'stdp_synapse',
    'alpha': 1.0,
    'lambda': 0.01,
    'mu_plus': 1.0,
    'mu_minus': 1.0,
    'tau_plus': 20.0,
    'Wmax': 100.0,
    'weight': 20.0,
    'delay': 1.5
}

# Create STDP connections
nest.Connect(neurons_stdp, neurons_stdp,
             conn_spec={'rule': 'pairwise_bernoulli', 'p': p_conn},
             syn_spec=stdp_params)

connections_stdp = nest.GetConnections(neurons_stdp, neurons_stdp)

# Store initial weights
initial_weights = np.array(connections_stdp.get('weight'))

print(f"Created {len(connections_stdp)} STDP synapses")
print(f"  - Initial mean weight: {np.mean(initial_weights):.2f} pA")

In [ ]:
# Create recording devices
spike_recorder_stdp = nest.Create('spike_recorder')
nest.Connect(neurons_stdp, spike_recorder_stdp)

print("Recording devices created")

In [ ]:
# Run STDP simulation (longer)
sim_time_stdp = 2000.0
print(f"Simulating {sim_time_stdp} ms with STDP...")
nest.Simulate(sim_time_stdp)
print("STDP simulation complete!")

In [ ]:
# Extract spike data
spike_events_stdp = spike_recorder_stdp.get('events')
spike_times_stdp = spike_events_stdp['times']
spike_senders_stdp = spike_events_stdp['senders']

total_spikes_stdp = len(spike_times_stdp)
mean_firing_rate_stdp = total_spikes_stdp / (N_neurons * sim_time_stdp / 1000.0)

# Get final weights
final_weights = np.array(connections_stdp.get('weight'))
weight_changes = final_weights - initial_weights

print(f"\nSTDP Network Statistics:")
print(f"  - Total spikes: {total_spikes_stdp}")
print(f"  - Mean firing rate: {mean_firing_rate_stdp:.2f} Hz")
print(f"\nWeight Evolution:")
print(f"  - Initial mean: {np.mean(initial_weights):.2f} pA")
print(f"  - Final mean: {np.mean(final_weights):.2f} pA")
print(f"  - Mean change: {np.mean(weight_changes):.2f} pA")
print(f"  - Std of changes: {np.std(weight_changes):.2f} pA")

In [ ]:
# Plot STDP analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Raster plot
axes[0, 0].scatter(spike_times_stdp, spike_senders_stdp, s=1, c='black', alpha=0.5)
axes[0, 0].set_ylabel('Neuron ID', fontsize=12)
axes[0, 0].set_xlabel('Time (ms)', fontsize=12)
axes[0, 0].set_title('Network Activity with STDP', fontsize=14, fontweight='bold')
axes[0, 0].set_xlim(0, sim_time_stdp)
axes[0, 0].grid(True, alpha=0.3)

# 2. Population firing rate
bin_size = 20.0
bins = np.arange(0, sim_time_stdp + bin_size, bin_size)
hist, _ = np.histogram(spike_times_stdp, bins=bins)
firing_rate_pop_stdp = hist / (N_neurons * bin_size / 1000.0)
bin_centers = (bins[:-1] + bins[1:]) / 2

axes[0, 1].plot(bin_centers, firing_rate_pop_stdp, 'b-', linewidth=2)
axes[0, 1].axhline(mean_firing_rate_stdp, color='r', linestyle='--',
                   label=f'Mean = {mean_firing_rate_stdp:.2f} Hz', linewidth=2)
axes[0, 1].set_ylabel('Firing Rate (Hz)', fontsize=12)
axes[0, 1].set_xlabel('Time (ms)', fontsize=12)
axes[0, 1].set_title('Population Firing Rate', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Weight changes
axes[1, 0].hist(weight_changes, bins=30, color='purple', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(0, color='r', linestyle='--', linewidth=2, label='No change')
axes[1, 0].axvline(np.mean(weight_changes), color='g', linestyle='--',
                   linewidth=2, label=f'Mean = {np.mean(weight_changes):.2f} pA')
axes[1, 0].set_xlabel('Weight Change (pA)', fontsize=12)
axes[1, 0].set_ylabel('Count', fontsize=12)
axes[1, 0].set_title('Distribution of Weight Changes', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Final weight distribution
axes[1, 1].hist(final_weights, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(np.mean(final_weights), color='r', linestyle='--',
                   linewidth=2, label=f'Final = {np.mean(final_weights):.2f} pA')
axes[1, 1].axvline(np.mean(initial_weights), color='g', linestyle='--',
                   linewidth=2, label=f'Initial = {np.mean(initial_weights):.2f} pA')
axes[1, 1].set_xlabel('Synaptic Weight (pA)', fontsize=12)
axes[1, 1].set_ylabel('Count', fontsize=12)
axes[1, 1].set_title('Final Weight Distribution', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Figure 2: STDP analysis complete!")

---
## Part 4: Connectivity Comparison

In [ ]:
def simulate_network(connection_prob, sim_duration=1000.0):
    """Simulate network with given connectivity"""
    nest.ResetKernel()
    nest.resolution = 0.1
    
    # Create neurons
    neurons = nest.Create('iaf_psc_alpha', N_neurons, params=neuron_params)
    
    # Poisson input
    poisson_gen = nest.Create('poisson_generator', N_neurons, params={'rate': POISSON_RATE})
    nest.Connect(poisson_gen, neurons, 'one_to_one',
                 syn_spec={'weight': INPUT_WEIGHT, 'delay': 1.0})
    
    # Recurrent connections
    nest.Connect(neurons, neurons,
                 conn_spec={'rule': 'pairwise_bernoulli', 'p': connection_prob},
                 syn_spec={'weight': RECURRENT_WEIGHT, 'delay': 1.5})
    
    # Spike recorder
    spike_rec = nest.Create('spike_recorder')
    nest.Connect(neurons, spike_rec)
    
    # Simulate
    nest.Simulate(sim_duration)
    
    # Extract
    events = spike_rec.get('events')
    return events['times'], events['senders']

In [ ]:
# Simulate sparse and dense networks
print("Simulating sparse network (p=0.1)...")
times_sparse, senders_sparse = simulate_network(0.1)
rate_sparse = len(times_sparse) / (N_neurons * 1.0)

print("Simulating dense network (p=0.3)...")
times_dense, senders_dense = simulate_network(0.3)
rate_dense = len(times_dense) / (N_neurons * 1.0)

print(f"\nResults:")
print(f"  - Sparse (p=0.1): {rate_sparse:.2f} Hz")
print(f"  - Dense (p=0.3): {rate_dense:.2f} Hz")
print(f"  - Increase: {(rate_dense/rate_sparse - 1)*100:.1f}%")

In [ ]:
# Plot connectivity comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Sparse raster
axes[0, 0].scatter(times_sparse, senders_sparse, s=1, c='black', alpha=0.5)
axes[0, 0].set_ylabel('Neuron ID', fontsize=12)
axes[0, 0].set_xlabel('Time (ms)', fontsize=12)
axes[0, 0].set_title(f'Sparse Network (p=0.1)\nRate: {rate_sparse:.2f} Hz',
                     fontsize=14, fontweight='bold')
axes[0, 0].set_xlim(0, 1000)
axes[0, 0].grid(True, alpha=0.3)

# Dense raster
axes[0, 1].scatter(times_dense, senders_dense, s=1, c='black', alpha=0.5)
axes[0, 1].set_ylabel('Neuron ID', fontsize=12)
axes[0, 1].set_xlabel('Time (ms)', fontsize=12)
axes[0, 1].set_title(f'Dense Network (p=0.3)\nRate: {rate_dense:.2f} Hz',
                     fontsize=14, fontweight='bold')
axes[0, 1].set_xlim(0, 1000)
axes[0, 1].grid(True, alpha=0.3)

# Sparse population activity
bin_size = 10.0
bins = np.arange(0, 1000 + bin_size, bin_size)
hist_sparse, _ = np.histogram(times_sparse, bins=bins)
rate_pop_sparse = hist_sparse / (N_neurons * bin_size / 1000.0)
bin_centers = (bins[:-1] + bins[1:]) / 2

axes[1, 0].plot(bin_centers, rate_pop_sparse, 'b-', linewidth=2)
axes[1, 0].set_ylabel('Firing Rate (Hz)', fontsize=12)
axes[1, 0].set_xlabel('Time (ms)', fontsize=12)
axes[1, 0].set_title('Sparse - Population Activity', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Dense population activity
hist_dense, _ = np.histogram(times_dense, bins=bins)
rate_pop_dense = hist_dense / (N_neurons * bin_size / 1000.0)

axes[1, 1].plot(bin_centers, rate_pop_dense, 'r-', linewidth=2)
axes[1, 1].set_ylabel('Firing Rate (Hz)', fontsize=12)
axes[1, 1].set_xlabel('Time (ms)', fontsize=12)
axes[1, 1].set_title('Dense - Population Activity', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Figure 3 complete!")

---
## Part 5: Synchronization Analysis

In [ ]:
def calculate_synchrony(spike_times, spike_senders, bin_size=5.0, sim_time=1000.0):
    bins = np.arange(0, sim_time + bin_size, bin_size)
    hist, _ = np.histogram(spike_times, bins=bins)
    synchrony = np.var(hist) / np.mean(hist) if np.mean(hist) > 0 else 0
    return synchrony, hist

def calculate_cv_isi(spike_times, spike_senders, neuron_ids):
    cvs = []
    for nid in neuron_ids:
        neuron_spikes = spike_times[spike_senders == nid]
        if len(neuron_spikes) > 1:
            isis = np.diff(neuron_spikes)
            if np.mean(isis) > 0:
                cv = np.std(isis) / np.mean(isis)
                cvs.append(cv)
    return np.mean(cvs) if cvs else 0

# Calculate metrics
sync_sparse, hist_sparse = calculate_synchrony(times_sparse, senders_sparse)
sync_dense, hist_dense = calculate_synchrony(times_dense, senders_dense)

neuron_ids = np.arange(1, N_neurons + 1)
cv_sparse = calculate_cv_isi(times_sparse, senders_sparse, neuron_ids)
cv_dense = calculate_cv_isi(times_dense, senders_dense, neuron_ids)

print(f"\nSynchronization Analysis:")
print(f"\n  Sparse (p=0.1):")
print(f"    - Synchrony: {sync_sparse:.3f}")
print(f"    - CV of ISI: {cv_sparse:.3f}")
print(f"\n  Dense (p=0.3):")
print(f"    - Synchrony: {sync_dense:.3f}")
print(f"    - CV of ISI: {cv_dense:.3f}")

In [ ]:
# Plot synchronization comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar plot
measures = ['Synchrony\nIndex', 'CV of\nISI']
sparse_vals = [sync_sparse, cv_sparse]
dense_vals = [sync_dense, cv_dense]

x = np.arange(len(measures))
width = 0.35

bars1 = axes[0].bar(x - width/2, sparse_vals, width, label='Sparse (p=0.1)',
                    color='steelblue', edgecolor='black', linewidth=1.5)
bars2 = axes[0].bar(x + width/2, dense_vals, width, label='Dense (p=0.3)',
                    color='coral', edgecolor='black', linewidth=1.5)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}', ha='center', va='bottom', fontsize=10)

axes[0].set_ylabel('Value', fontsize=12)
axes[0].set_title('Synchronization Measures', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(measures)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Spike count distributions
axes[1].hist(hist_sparse, bins=20, alpha=0.6, label='Sparse (p=0.1)',
             color='steelblue', edgecolor='black')
axes[1].hist(hist_dense, bins=20, alpha=0.6, label='Dense (p=0.3)',
             color='coral', edgecolor='black')
axes[1].set_xlabel('Spike Count per Bin', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Spike Count Distributions', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Figure 4 complete!")